# Backward particle tracking

This example demonstrates backward particle tracking. The scenario represents a common backwards tracking use case: determining the capture area for a pumping well. The flow system is taken from [an example](https://modflow6-examples.readthedocs.io/en/develop/_notebooks/ex-prt-mp7-p01.html) originally published in the MODPATH 7 user guide.

We will first create and run a flow model in its own simulation. We will then demonstrate equivalent MODPATH 7 and MODFLOW 6 PRT models consuming the flow model's results. The idea here is to show how to migrate MP7 models to PRT and compare and contrast the two particle tracking codes.

First, define units.

In [ ]:
time_unit = "days"
length_unit = "feet"

Set up time discretization information. This simulation has a single stress period and time step.

In [ ]:
nstp = 1
perlen = 1000.0
tsmult = 1.0
tdis_pd = [(perlen, nstp, tsmult)]
nper = len(tdis_pd)

Set up  grid discretization info. This is a three-layer model with two relatively permeable layers sandwiching a narrow confining bed.

In [ ]:
nlay, nrow, ncol = 3, 21, 20
Lx, Ly = 10000.0, 10500.0
delr, delc = Lx / ncol, Ly / nrow
top = 400.0
bot = [220.0, 200.0, 0.0]

Set up a base workspace for the example scenario, and a nested workspace for the flow model.

In [ ]:
from pathlib import Path

example_name = "prt_backward"
gwf_name = f"{example_name}-gwf"
base_ws = Path("models") / example_name
gwf_ws = base_ws / "gwf"
gwf_ws.mkdir(exist_ok=True, parents=True)

Create the flow simulation and model, starting with time and grid discretizations, then initial conditions.

In [ ]:
import flopy

gwf_sim = flopy.mf6.MFSimulation(
    sim_name=gwf_name,
    exe_name="mf6",
    version="mf6",
    sim_ws=gwf_ws,
)
tdis = flopy.mf6.ModflowTdis(
    gwf_sim,
    time_units=time_unit,
    nper=nper,
    perioddata=tdis_pd,
)
gwf = flopy.mf6.ModflowGwf(
    gwf_sim,
    modelname=gwf_name,
    save_flows=True,
)
dis = flopy.mf6.ModflowGwfdis(
    gwf,
    length_units=length_unit,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=bot,
)
ic = flopy.mf6.ModflowGwfic(gwf, strt=top)

Set up the node property flow package. Cells in the model grid's top layer are convertible. The bottom two layers are confined. Hydraulic conductivity is greater in the lateral than the vertical direction.

In [ ]:
kh = [50.0, 0.01, 200.0]
kv = [10.0, 0.01, 20.0]
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    icelltype=[1, 0, 0],
    k=kh,
    k33=kv,
    save_flows=True,
    save_saturation=True,
    save_specific_discharge=True,
)

Define boundary conditions, starting with a pumping well in the model's bottom layer.

In [ ]:
wel_q = -150000.0
wel_loc = (2, 10, 9)
wel_pd = [(wel_loc, wel_q)]
well = flopy.mf6.modflow.mfgwfwel.ModflowGwfwel(
    gwf, maxbound=1, stress_period_data={0: wel_pd}
)

A river runs along the model grid's right edge. This presents the first opportunity to compare MODPATH 7 and PRT. By default, boundary conditions function as distributed sources or sinks. To assign boundary flows to a particular cell face, we can use MF6's auxiliary variable mechanism. MODPATH 7 supports this via an auxiliary variable called `IFACE`. PRT supports the same thing, but calls it `IFLOWFACE`, and uses a different face numbering scheme. Here we configure the flow through the river cells' top face.

In [ ]:
riv_h = 320.0
riv_z = 317.0
riv_c = 1.0e5
riv_iface = 6
riv_iflowface = -1
riv_pd = []
for i in range(nrow):
    riv_pd.append(
        [
            # (layer, row, column), stage, conductance, bottom, [aux, ...]
            (0, i, ncol - 1),
            riv_h,
            riv_c,
            riv_z,
            riv_iface,
            riv_iflowface,
        ]
    )
riv = flopy.mf6.modflow.mfgwfriv.ModflowGwfriv(
    gwf, auxiliary=["iface", "iflowface"], stress_period_data={0: riv_pd}
)

Define recharge across the entire top of the model. Note the same `IFACE`/`IFLOWFACE` values as for the river.

In [ ]:
rch = 0.005
rch_iface = 6
rch_iflowface = -1
rcha = flopy.mf6.modflow.mfgwfrcha.ModflowGwfrcha(
    gwf,
    recharge=rch,
    auxiliary=["iface", "iflowface"],
    aux=[rch_iface, rch_iflowface],
)

Configure output control, enabling comprehensive head and budget reporting.

In [ ]:
headfile_name = f"{gwf_name}.hds"
budgetfile_name = f"{gwf_name}.cbb"

oc = flopy.mf6.modflow.mfgwfoc.ModflowGwfoc(
    gwf,
    pname="oc",
    head_filerecord=[headfile_name],
    budget_filerecord=[budgetfile_name],
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
)

Create an iterative model solution.

In [ ]:
ims = flopy.mf6.ModflowIms(
    gwf_sim,
    pname="ims",
    complexity="SIMPLE",
    outer_dvclose=1e-6,
    inner_dvclose=1e-6,
    rcloserecord=1e-6,
)

Write and run the flow simulation.

In [ ]:
gwf_sim.write_simulation(silent=False)
gwf_sim.run_simulation(silent=False)

Load flow results, then plot heads in the bottom layer.

In [ ]:
hds = gwf.output.head()
head = hds.get_data()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(6, 6))
    ax.set_aspect("equal")
    ax.set_title("Head, layer 3", fontsize=10)
    ax.legend(
        handles=[
            Patch(color="red", label="WELL"),
            Patch(color="blue", label="RIV"),
        ],
        loc="upper left",
    )
    mm = flopy.plot.PlotMapView(gwf, ax=ax, layer=2)
    mm.plot_grid(alpha=0.25)
    mm.plot_bc("WELL", plotAll=True, color="red")
    mm.plot_bc("RIV", plotAll=True, color="blue")
    pc = mm.plot_array(head, alpha=0.25)
    cb = plt.colorbar(pc, shrink=0.25, pad=0.1)
    cb.ax.set_xlabel(r"Head (ft)")
    fig.tight_layout()
    plt.show()

We can now begin setting up the particle tracking models, starting with MODPATH 7.

Release locations are again defined with flopy's MP7 release-template module, then converted for PRT.  We release particles from the lateral faces of the well cell using MP7 particle input style 2,
subdivision style 1.

In [ ]:
mp7_face_data = flopy.modpath.FaceDataType(
    verticaldivisions1=5,
    verticaldivisions2=5,
    verticaldivisions3=5,
    verticaldivisions4=5,
    horizontaldivisions1=5,
    horizontaldivisions2=5,
    horizontaldivisions3=5,
    horizontaldivisions4=5,
    rowdivisions5=0,
    rowdivisions6=0,
    columndivisions5=0,
    columndivisions6=0,
)
mp7_particle_data = flopy.modpath.LRCParticleData(
    subdivisiondata=[mp7_face_data], lrcregions=[[[*wel_loc, *wel_loc]]]
)
mp7_pg1 = flopy.modpath.ParticleGroupLRCTemplate(
    particlegroupname="PG1",
    particledata=mp7_particle_data,
    filename="pg1.sloc",
)

Set up the MP7 simulation.

In [ ]:
mp7_name = f"{example_name}-mp7"
mp7_ws = base_ws / "mp7"
mp7_ws.mkdir(exist_ok=True, parents=True)

mp7 = flopy.modpath.Modpath7(
    modelname=mp7_name,
    flowmodel=gwf,
    exe_name="mp7",
    model_ws=mp7_ws,
    budgetfilename=budgetfile_name,
    headfilename=headfile_name,
)

porosity = 0.1
mp7_bas = flopy.modpath.Modpath7Bas(
    mp7,
    porosity=porosity,
    defaultiface={"RCH": 6},  # TODO is this necessary??
)

mp7_sim = flopy.modpath.Modpath7Sim(
    mp7,
    simulationtype="pathline",
    trackingdirection="backward",
    budgetoutputoption="summary",
    stoptimeoption="extend",
    particlegroups=[mp7_pg1],
)

Write and run the MP7 simulation.

In [ ]:
mp7.write_input()
mp7.run_model(silent=False)

Load and plot MP7 pathlines. FloPy's MP7 `PathlineFile` utility returns a list of recarrays, one for each particle. Below we merge these into a single dataframe for easier inspection.

In [ ]:
import pandas as pd

mp7_pathline_file = flopy.utils.PathlineFile(mp7_ws / f"{mp7_name}.mppth")
mp7_pathlines = pd.concat(
    [pd.DataFrame(pdata) for pdata in mp7_pathline_file.get_alldata()]
)
mp7_pathlines

Plot MP7 pathlines.

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(10, 5))
    ax.set_aspect("equal")
    ax.set_title("MODPATH 7 pathlines", fontsize=11)
    ax.legend(
        handles=[
            Patch(color="red", label="WELL"),
            Patch(color="blue", label="RIV"),
        ],
        loc="upper left",
    )
    mm = flopy.plot.PlotMapView(gwf, ax=ax, layer=2)
    mm.plot_grid(alpha=0.2)
    mm.plot_bc("WELL", plotAll=True, color="red")
    mm.plot_bc("RIV", plotAll=True, color="blue")
    pc = mm.plot_array(head, alpha=0.25)
    cb = plt.colorbar(pc, shrink=0.25, pad=0.1)
    mm.plot_pathline(mp7_pathlines, layer="all")
    cb.ax.set_xlabel(r"Head (ft)")
    fig.tight_layout()
    plt.show()

Now construct an equivalent PRT simulation.

In [ ]:
prt_name = f"{example_name}-prt"
prt_ws = base_ws / "prt"
prt_ws.mkdir(exist_ok=True, parents=True)

prt_sim = flopy.mf6.MFSimulation(
    sim_name=prt_name, exe_name="mf6", version="mf6", sim_ws=prt_ws
)

tdis = flopy.mf6.modflow.mftdis.ModflowTdis(
    prt_sim,
    pname="tdis",
    time_units="DAYS",
    nper=nper,
    perioddata=[(perlen, nstp, tsmult)],
)

prt = flopy.mf6.ModflowPrt(
    prt_sim, modelname=prt_name, model_nam_file=f"{prt_name}.name"
)

dis = flopy.mf6.modflow.mfgwfdis.ModflowGwfdis(
    prt,
    pname="dis",
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    length_units="FEET",
    delr=delr,
    delc=delc,
    top=top,
    botm=bot,
)

mip = flopy.mf6.ModflowPrtmip(prt, pname="mip", porosity=porosity)

FloPy provides convenience functions to convert MP7 particle release templates to a form suitable for PRT. PRT does not (yet) support release templates like MP7. Each distinct release point must be explicitly specified.

In [ ]:
release_pts = list(mp7_particle_data.to_prp(gwf.modelgrid))
prp = flopy.mf6.ModflowPrtprp(
    prt,
    nreleasepts=len(release_pts),
    packagedata=release_pts,
    perioddata={0: ["FIRST"]},
    exit_solve_tolerance=1e-5,
    extend_tracking=True,
)

Create the output control package. PRT can write pathline output to binary or CSV files. We will just use CSV.

Where MODPATH 7 provides four distinct simulation "modes" (pathline, timeseries, endpoint, and combined), PRT has a single mode, and instead defines "tracking events" which the user may decide whether to enable or disable. Each of these is associated with a "reason code" (`ireason` column in the CSV output) identifying the event.

- 0: a particle was released
- 1: a particle exited a grid feature (e.g. a cell)
- 2: a time step ended
- 3: a particle terminated
- 4: a particle entered a weak sink cell (a cell that removes some but not all inflow)
- 5: the simulation reached a user-specified tracking time
- 6: a particle was dropped to the water table (more on this in the next notebook)

By default, reporting is enabled for all events. If any events are explicitly enabled, other events will not be reported.

Q: How to configure a PRT model like an MP7 endpoint simulation?
A: Select release and termination events only.

Q: How to configure a model like an MP7 timeseries simulation?
A: Select time reporting and provide tracking times.

In [ ]:
budgetfile_prt = f"{prt_name}.cbb"
trackcsvfile_prt = f"{prt_name}.trk.csv"

oc = flopy.mf6.ModflowPrtoc(
    prt,
    pname="oc",
    budget_filerecord=[budgetfile_prt],
    trackcsv_filerecord=[trackcsvfile_prt],
    saverecord=[("BUDGET", "ALL")],
)

Create the flow model interface. PRT does not (yet) natively support backward tracking, so we reverse the flow model's head and budget files with FloPy, reversing the time order of records and switching the sign of flows.

In [ ]:
head_file = flopy.utils.HeadFile(gwf_ws / headfile_name, tdis=gwf_sim.tdis)
budget_file = flopy.utils.CellBudgetFile(
    gwf_ws / budgetfile_name, precision="double", tdis=gwf_sim.tdis
)

headfile_bkwd_name = f"{headfile_name}_bkwd"
budgetfile_bkwd_name = f"{budgetfile_name}_bkwd"

head_file.reverse(prt_ws / headfile_bkwd_name)
budget_file.reverse(prt_ws / budgetfile_bkwd_name)

fmi = flopy.mf6.ModflowPrtfmi(
    prt,
    packagedata=[
        ("GWFHEAD", headfile_bkwd_name),
        ("GWFBUDGET", budgetfile_bkwd_name),
    ],
)

Create an explicit model solution. PRT uses its own explicit solution procedure, not the iterative matrix solvers used by flow and transport models.

In [ ]:
ems = flopy.mf6.ModflowEms(
    prt_sim,
    pname="ems",
    filename=f"{prt_name}.ems",
)
prt_sim.register_solution_package(ems, [prt.name])

Write and run the PRT simulation.

In [ ]:
prt_sim.write_simulation()
prt_sim.run_simulation(silent=False)

Load PRT pathlines.

In [ ]:
prt_pathlines = pd.read_csv(prt_ws / trackcsvfile_prt)
prt_pathlines

Compare PRT and MP7 pathlines in 3D with PyVista.

In [ ]:
import pyvista as pv
from flopy.export.vtk import Vtk

vert_exag = 10

vtk = Vtk(model=gwf, binary=False, vertical_exageration=vert_exag, smooth=False)
vtk.add_model(gwf)
vtk.add_pathline_points(mp7_pathlines.to_records(index=False))
gwf_mesh, mp7_mesh = vtk.to_pyvista()

# FloPy's Vtk class only supports one pathline set at a time, so rebuild for PRT
vtk = Vtk(model=gwf, binary=False, vertical_exageration=vert_exag, smooth=False)
vtk.add_model(gwf)
vtk.add_pathline_points(prt_pathlines.to_records(index=False))
_, prt_mesh = vtk.to_pyvista()


def get_nn(k, i, j):
    return k * nrow * ncol + i * ncol + j


riv_mesh = pv.Box(
    bounds=[
        gwf.modelgrid.extent[1] - delc,
        gwf.modelgrid.extent[1],
        gwf.modelgrid.extent[2],
        gwf.modelgrid.extent[3],
        bot[0] * vert_exag,
        head[(0, 0, ncol - 1)] * vert_exag,
    ]
)
well_cellid = get_nn(0, *wel_loc[1:])
well_points = gwf.modelgrid.verts[gwf.modelgrid.iverts[well_cellid]]
well_xs, well_ys = list(zip(*well_points, strict=False))
wel_mesh = pv.Box(
    bounds=[
        min(well_xs),
        max(well_xs),
        min(well_ys),
        max(well_ys),
        bot[-1] * vert_exag,
        bot[-2] * vert_exag,
    ]
)
bed_mesh = pv.Box(
    bounds=[
        gwf.modelgrid.extent[0],
        gwf.modelgrid.extent[1],
        gwf.modelgrid.extent[2],
        gwf.modelgrid.extent[3],
        bot[1] * vert_exag,
        bot[0] * vert_exag,
    ]
)

pv.set_jupyter_backend("static")
p = pv.Plotter(window_size=[500, 500], notebook=True)
p.enable_anti_aliasing()
p.add_mesh(gwf_mesh, opacity=0.025, style="wireframe")
p.add_mesh(mp7_mesh, point_size=5, line_width=2.5, smooth_shading=True, color="purple")
p.add_mesh(prt_mesh, point_size=5, line_width=2.5, smooth_shading=True, color="pink")
p.add_mesh(riv_mesh, color="teal", opacity=0.2)
p.add_mesh(wel_mesh, color="red", opacity=0.3)
p.add_mesh(bed_mesh, color="tan", opacity=0.1)
p.camera.elevation -= 15
p.show()

To estimate the area of the well's capture zone, we can compute the convex hull of the final particle positions. First, extract termination events.

In [ ]:
terminals = prt_pathlines[prt_pathlines.ireason == 3]
assert len(terminals) == len(release_pts)

Build a multi-point geometry and compute the convex hull.

In [ ]:
from shapely import MultiPoint, convex_hull

term_pts = MultiPoint(terminals[["x", "y", "z"]].to_numpy())
chull = convex_hull(term_pts)
chull_area_mi = chull.area / (5280 * 5280)
print(f"Capture area: {chull.area:9.3f} sq {length_unit}, {chull_area_mi:9.3f} sq mi")
chull

Plot pathlines with the capture zone overlaid.

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(6, 6))
    ax.set_aspect("equal")
    fig.tight_layout()
    ax.set_title("PRT pathlines with capture zone", fontsize=11)
    ax.legend(
        handles=[
            Patch(color="red", label="WELL"),
            Patch(color="blue", label="RIV"),
            Patch(color="teal", label="capture zone"),
        ],
        loc="upper left",
    )
    mm = flopy.plot.PlotMapView(gwf, ax=ax, layer=2)
    mm.plot_grid(alpha=0.2)
    mm.plot_bc("WELL", plotAll=True, color="red")
    mm.plot_bc("RIV", plotAll=True, color="blue")
    pc = mm.plot_array(head, alpha=0.25)
    mm.plot_pathline(prt_pathlines, layer="all")
    chull_xs, chull_ys, _ = map(list, zip(*list(chull.exterior.coords), strict=False))
    ax.fill(chull_xs, chull_ys, "teal", alpha=0.5)
    ax.annotate(f"{chull_area_mi:.3f} sq mi", (1000, 3000), color="red")
    plt.show()

Now, we can determine a travel time distribution.

In [ ]:
terminals.t.describe()

In [ ]:
terminals.t.plot(kind="hist")
plt.show()

Most particles travel from the surface to the well in under ~30 years.

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(6, 6))
    ax.set_aspect("equal")
    fig.tight_layout()
    ax.set_title("PRT endpoints colored by travel time", fontsize=11)
    ax.legend(
        handles=[
            Patch(color="red", label="WELL"),
            Patch(color="blue", label="RIV"),
        ],
        loc="upper left",
    )
    mm = flopy.plot.PlotMapView(gwf, ax=ax, layer=2)
    mm.plot_grid(alpha=0.2)
    mm.plot_bc("WELL", plotAll=True, color="red")
    mm.plot_bc("RIV", plotAll=True, color="blue")
    pc = mm.plot_array(head, alpha=0.25)
    pc = ax.scatter(terminals.x, terminals.y, c=terminals.t)
    cb = plt.colorbar(pc, shrink=0.25, pad=0.1)
    cb.ax.set_xlabel(r"Travel time ($d$)")
    plt.show()